# SMAP soil moisture (NASA Earthdata)

Download a daily SMAP enhanced L3 radiometer soil-moisture granule from the
NSIDC DAAC and map the morning (AM) retrieval. The granule is an HDF5 **raster**
on the EASE-2 grid.

> This is a live query — it needs the `[earthdata]` extra and Earthdata Login
> (EDL) credentials. Each step is wrapped in a `try`/`except` so the notebook
> stays nbval-lax safe and skips gracefully offline.

## Setup

Imports and an output directory for the downloaded granule. `earthlens` provides
the unified `EarthLens` entry point; `Path` gives us a tidy place to write the
HDF5 file.

In [ ]:
from pathlib import Path

from earthlens.core import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

## Download the granule

### Build the request

We describe the granule we want — the `SPL3SMP_E_006` enhanced L3 product, the
`soil_moisture` variable, a single day, and a global bounding box. Earthdata is a
**raster** backend, so we use `download()` to write the granule to disk.

In [ ]:
smap = EarthLens(
    data_source='earthdata',
    dataset='SPL3SMP_E_006',
    variables=['soil_moisture'],
    start='2023-06-01',
    end='2023-06-01',
    aoi=[-180.0, -90.0, 180.0, 90.0],
    path=str(OUT_DIR),
)

### Fetch it

`download()` runs the live NSIDC query and returns the list of written paths. The
whole call is guarded so a missing `[earthdata]` extra or absent EDL credentials
just prints a skip message instead of raising.

In [ ]:
paths = None
try:
    paths = smap.download(progress_bar=False)
    print(len(paths), 'granule(s):', [Path(p).name for p in paths])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Map the AM retrieval

If the download succeeded, read the morning (AM) soil-moisture array out of the
HDF5 granule, mask the fill values, and map it. This step is wrapped too, so it
skips cleanly when the granule is missing or the plotting deps are unavailable.

In [ ]:
if paths:
    try:
        import h5py
        import matplotlib.pyplot as plt
        import numpy as np

        with h5py.File(paths[0], 'r') as h5:
            sm = h5['Soil_Moisture_Retrieval_Data_AM/soil_moisture'][:]
        sm = np.where(sm < 0, np.nan, sm)
        fig, ax = plt.subplots(figsize=(10, 5))
        im = ax.imshow(sm, cmap='YlGnBu', vmin=0, vmax=0.5)
        fig.colorbar(im, ax=ax, label='Soil moisture (cm3/cm3)')
        ax.set_title('SMAP enhanced L3 soil moisture (AM), 2023-06-01')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'read/plot step skipped: {type(exc).__name__}: {exc}')